In [ ]:
!pip -q install fiftyone "eta<1.0.0" tqdm pyyaml

  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.8/112.8 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.5/112.5 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.8/74.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 83.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.4/316.4 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 934.5/934.5 kB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import os
import shutil
import random
import glob
from collections import Counter, defaultdict
from pathlib import Path

import yaml
from tqdm.auto import tqdm

import fiftyone as fo
import fiftyone.zoo as foz
from fiftyone import ViewField as F

/usr/local/lib/python3.12/dist-packages/glob2/fnmatch.py:141: SyntaxWarning: invalid escape sequence '\Z'
  return '(?ms)' + res + '\Z'


In [ ]:
BASE_DIR = "/content/drive/MyDrive/Lab thầy Vinh/Train_VOC_Test_Dawn/Train_VOC_Test_Dawn"
YAML_PATH = os.path.join(BASE_DIR, "train_voc_test_dawn.yaml")

TRAIN_MULTIPLIER = 3.0
VAL_MULTIPLIER = 3.0

RANDOM_SEED = 42
BACKUP_ORIGINAL_FOLDERS = True
OVERWRITE_EXISTING_EXPORT_TMP = True

TMP_DIR = "/content/openimages_expansion_tmp"
OI_EXPORT_DIR = os.path.join(TMP_DIR, "oi_yolo_export")

random.seed(RANDOM_SEED)

In [ ]:
with open(YAML_PATH, "r", encoding="utf-8") as f:
    data_yaml = yaml.safe_load(f)

print("Loaded YAML:")
print(data_yaml)

required_names = {
    0: "person",
    1: "bicycle",
    2: "car",
    3: "motorcycle",
    4: "bus",
    5: "train",
}

assert data_yaml["names"] == required_names, f"Unexpected class mapping: {data_yaml['names']}"

train_img_dir = os.path.join(BASE_DIR, "train", "images")
train_lbl_dir = os.path.join(BASE_DIR, "train", "labels")
val_img_dir = os.path.join(BASE_DIR, "val", "images")
val_lbl_dir = os.path.join(BASE_DIR, "val", "labels")
test_img_dir = os.path.join(BASE_DIR, "test", "images")
test_lbl_dir = os.path.join(BASE_DIR, "test", "labels")

for d in [train_img_dir, train_lbl_dir, val_img_dir, val_lbl_dir, test_img_dir]:
    os.makedirs(d, exist_ok=True)

if os.path.exists(test_lbl_dir):
    os.makedirs(test_lbl_dir, exist_ok=True)

Loaded YAML:
{'path': '/content/drive/MyDrive/Lab thầy Vinh/Train_VOC_Test_Dawn/Train_VOC_Test_Dawn', 'train': 'train/images', 'val': 'val/images', 'test': 'test/images', 'names': {0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 4: 'bus', 5: 'train'}}


In [ ]:
def count_images_labels(img_dir, lbl_dir):
    img_exts = ("*.jpg", "*.jpeg", "*.png", "*.bmp", "*.webp")
    images = []
    for ext in img_exts:
        images.extend(glob.glob(os.path.join(img_dir, ext)))
    labels = glob.glob(os.path.join(lbl_dir, "*.txt"))
    return len(images), len(labels)

current_train_images, current_train_labels = count_images_labels(train_img_dir, train_lbl_dir)
current_val_images, current_val_labels = count_images_labels(val_img_dir, val_lbl_dir)

if os.path.exists(test_lbl_dir):
    current_test_images, current_test_labels = count_images_labels(test_img_dir, test_lbl_dir)
else:
    current_test_images = count_images_labels(test_img_dir, test_img_dir)[0]
    current_test_labels = 0

print(f"Current train: {current_train_images} images, {current_train_labels} labels")
print(f"Current val  : {current_val_images} images, {current_val_labels} labels")
print(f"Current test : {current_test_images} images, {current_test_labels} labels")

target_train_total = int(round(current_train_images * TRAIN_MULTIPLIER))
target_val_total = int(round(current_val_images * VAL_MULTIPLIER))

need_add_train = max(0, target_train_total - current_train_images)
need_add_val = max(0, target_val_total - current_val_images)

print("\nTarget sizes:")
print(f"Train target total: {target_train_total}  -> need to add {need_add_train}")
print(f"Val   target total: {target_val_total}    -> need to add {need_add_val}")

if need_add_train == 0 and need_add_val == 0:
    raise ValueError("Nothing to add. Increase TRAIN_MULTIPLIER / VAL_MULTIPLIER.")

Current train: 4786 images, 4786 labels
Current val  : 1199 images, 1199 labels
Current test : 1015 images, 1015 labels

Target sizes:
Train target total: 14358  -> need to add 9572
Val   target total: 3597    -> need to add 2398


In [ ]:
def backup_folder(folder_path):
    backup_path = folder_path.rstrip("/\\") + "_backup_before_oi_expand"
    if os.path.exists(backup_path):
        print(f"Backup already exists: {backup_path}")
    else:
        shutil.copytree(folder_path, backup_path)
        print(f"Created backup: {backup_path}")

if BACKUP_ORIGINAL_FOLDERS:
    for folder in [train_img_dir, train_lbl_dir, val_img_dir, val_lbl_dir]:
        backup_folder(folder)

Created backup: /content/drive/MyDrive/Lab thầy Vinh/Train_VOC_Test_Dawn/Train_VOC_Test_Dawn/train/images_backup_before_oi_expand
Created backup: /content/drive/MyDrive/Lab thầy Vinh/Train_VOC_Test_Dawn/Train_VOC_Test_Dawn/train/labels_backup_before_oi_expand
Created backup: /content/drive/MyDrive/Lab thầy Vinh/Train_VOC_Test_Dawn/Train_VOC_Test_Dawn/val/images_backup_before_oi_expand
Created backup: /content/drive/MyDrive/Lab thầy Vinh/Train_VOC_Test_Dawn/Train_VOC_Test_Dawn/val/labels_backup_before_oi_expand


In [ ]:
OI_CLASSES = ["Person", "Bicycle", "Car", "Motorcycle", "Bus", "Train"]

download_train_limit = max(need_add_train * 3, 3000)
download_val_limit = max(need_add_val * 3, 1000)

print("\nDownloading Open Images V7 subsets with FiftyOne...")
print(f"Requested train max samples: {download_train_limit}")
print(f"Requested validation max samples: {download_val_limit}")

oi_train = foz.load_zoo_dataset(
    "open-images-v7",
    split="train",
    label_types=["detections"],
    classes=OI_CLASSES,
    max_samples=download_train_limit,
    shuffle=True,
    seed=RANDOM_SEED,
    dataset_name="oi_v7_train_for_expand",
)

oi_val = foz.load_zoo_dataset(
    "open-images-v7",
    split="validation",
    label_types=["detections"],
    classes=OI_CLASSES,
    max_samples=download_val_limit,
    shuffle=True,
    seed=RANDOM_SEED,
    dataset_name="oi_v7_val_for_expand",
)

print(f"Downloaded Open Images train subset: {len(oi_train)} samples")
print(f"Downloaded Open Images val subset  : {len(oi_val)} samples")


Requested train max samples: 28716
Requested validation max samples: 7194


INFO:fiftyone.zoo.datasets:Downloading split 'train' to '/root/fiftyone/open-images-v7/train' if necessary


INFO:fiftyone.utils.openimages:Downloading 'https://storage.googleapis.com/openimages/2018_04/train/train-images-boxable-with-rotation.csv' to '/root/fiftyone/open-images-v7/train/metadata/image_ids.csv'


 100% |██████|    4.8Gb/4.8Gb [6.6s elapsed, 0s remaining, 638.2Mb/s]       


INFO:eta.core.utils: 100% |██████|    4.8Gb/4.8Gb [6.6s elapsed, 0s remaining, 638.2Mb/s]       


...

 100% |███████████████| 7194/7194 [2.3m elapsed, 0s remaining, 63.4 samples/s]      


INFO:eta.core.utils: 100% |███████████████| 7194/7194 [2.3m elapsed, 0s remaining, 63.4 samples/s]      


Dataset 'oi_v7_val_for_expand' created


INFO:fiftyone.zoo.datasets:Dataset 'oi_v7_val_for_expand' created


Downloaded Open Images train subset: 28716 samples
Downloaded Open Images val subset  : 7194 samples


In [ ]:
def keep_only_target_classes(dataset, field_name="ground_truth"):
    allowed = list(OI_CLASSES)
    view = dataset.filter_labels(field_name, F("label").is_in(allowed))
    return view

oi_train_view = keep_only_target_classes(oi_train)
oi_val_view = keep_only_target_classes(oi_val)

print(f"Filtered train samples with target objects: {len(oi_train_view)}")
print(f"Filtered val samples with target objects  : {len(oi_val_view)}")

Filtered train samples with target objects: 28716
Filtered val samples with target objects  : 7194


In [ ]:
import os
import glob
from pathlib import Path

def get_image_stems(img_dir):
    stems = set()
    for ext in ("*.jpg", "*.jpeg", "*.png", "*.bmp", "*.webp"):
        for p in glob.glob(os.path.join(img_dir, ext)):
            stems.add(Path(p).stem)
    return stems

def get_label_stems(lbl_dir):
    return set(Path(p).stem for p in glob.glob(os.path.join(lbl_dir, "*.txt")))

def zero_byte_files(folder, patterns=("*.jpg", "*.jpeg", "*.png", "*.bmp", "*.webp", "*.txt")):
    bad = []
    for pat in patterns:
        for p in glob.glob(os.path.join(folder, pat)):
            try:
                if os.path.getsize(p) == 0:
                    bad.append(p)
            except Exception as e:
                bad.append(f"{p}  [stat failed: {e}]")
    return bad

def inspect_split(name, img_dir, lbl_dir):
    img_stems = get_image_stems(img_dir)
    lbl_stems = get_label_stems(lbl_dir)

    missing_labels = sorted(img_stems - lbl_stems)
    orphan_labels = sorted(lbl_stems - img_stems)

    zero_imgs = zero_byte_files(img_dir, ("*.jpg", "*.jpeg", "*.png", "*.bmp", "*.webp"))
    zero_lbls = zero_byte_files(lbl_dir, ("*.txt",))

    print(f"\n===== {name.upper()} =====")
    print(f"Images: {len(img_stems)}")
    print(f"Labels: {len(lbl_stems)}")
    print(f"Missing labels: {len(missing_labels)}")
    print(f"Orphan labels : {len(orphan_labels)}")
    print(f"Zero-byte images: {len(zero_imgs)}")
    print(f"Zero-byte labels: {len(zero_lbls)}")

    if missing_labels[:10]:
        print("Sample missing labels:", missing_labels[:10])
    if orphan_labels[:10]:
        print("Sample orphan labels:", orphan_labels[:10])
    if zero_imgs[:5]:
        print("Sample zero-byte images:", zero_imgs[:5])
    if zero_lbls[:5]:
        print("Sample zero-byte labels:", zero_lbls[:5])

    ok = (
        len(missing_labels) == 0 and
        len(orphan_labels) == 0 and
        len(zero_imgs) == 0 and
        len(zero_lbls) == 0
    )
    print(f"Status: {'OK' if ok else 'PROBLEM FOUND'}")
    return {
        "images": len(img_stems),
        "labels": len(lbl_stems),
        "missing_labels": missing_labels,
        "orphan_labels": orphan_labels,
        "zero_imgs": zero_imgs,
        "zero_lbls": zero_lbls,
        "ok": ok,
    }

# 1) File integrity on Drive
train_report = inspect_split("train", train_img_dir, train_lbl_dir)
val_report   = inspect_split("val", val_img_dir, val_lbl_dir)
test_report  = inspect_split("test", test_img_dir, test_lbl_dir if os.path.exists(test_lbl_dir) else test_img_dir)

# 2) Check test split has not changed
print("\n===== TEST UNCHANGED CHECK =====")
try:
    print("Current test images:", current_test_images)
    print("Current test labels:", current_test_labels)
    print("Observed test images:", test_report["images"])
    print("Observed test labels:", test_report["labels"])

    test_unchanged = (
        test_report["images"] == current_test_images and
        test_report["labels"] == current_test_labels
    )
    print("Test unchanged:", test_unchanged)
except Exception as e:
    print("Could not compare against earlier counts:", e)

# 3) Quick FiftyOne sanity check
print("\n===== FIFTYONE VIEW CHECK =====")
try:
    print("oi_train downloaded:", len(oi_train))
    print("oi_val downloaded  :", len(oi_val))
    print("oi_train_view kept :", len(oi_train_view))
    print("oi_val_view kept   :", len(oi_val_view))
    print("Cell 8 filtering status: OK")
except Exception as e:
    print("FiftyOne view check failed:", e)

# 4) Final overall verdict
all_ok = train_report["ok"] and val_report["ok"] and test_report["ok"]
print("\n===== OVERALL =====")
print("DATASET STATUS:", "SAFE TO CONTINUE" if all_ok else "STOP AND FIX BEFORE CONTINUING")


===== TRAIN =====
Images: 4786
Labels: 4786
Missing labels: 0
Orphan labels : 0
Zero-byte images: 0
Zero-byte labels: 0
Status: OK

===== VAL =====
Images: 1199
Labels: 1199
Missing labels: 0
Orphan labels : 0
Zero-byte images: 0
Zero-byte labels: 0
Status: OK

===== TEST =====
Images: 1015
Labels: 1015
Missing labels: 0
Orphan labels : 0
Zero-byte images: 0
Zero-byte labels: 0
Status: OK

===== TEST UNCHANGED CHECK =====
Current test images: 1015
Current test labels: 1015
Observed test images: 1015
Observed test labels: 1015
Test unchanged: True

===== FIFTYONE VIEW CHECK =====
oi_train downloaded: 28716
oi_val downloaded  : 7194
oi_train_view kept : 28716
oi_val_view kept   : 7194
Cell 8 filtering status: OK

===== OVERALL =====
DATASET STATUS: SAFE TO CONTINUE


In [ ]:
class_priority = {
    "Train": 6,
    "Bus": 5,
    "Motorcycle": 4,
    "Bicycle": 3,
    "Car": 2,
    "Person": 1,
}

def sample_balanced(view, n_needed, seed=42):
    samples = list(view)
    random.Random(seed).shuffle(samples)

    scored = []
    for s in samples:
        dets = s.ground_truth.detections if s.ground_truth is not None else []
        classes_present = set(d.label for d in dets if d.label in class_priority)
        if not classes_present:
            continue
        score = sum(class_priority[c] for c in classes_present)
        score += 2 * len(classes_present)
        scored.append((score, s))

    scored.sort(key=lambda x: x[0], reverse=True)
    chosen = [s for _, s in scored[:n_needed]]
    return chosen

selected_train_samples = sample_balanced(oi_train_view, need_add_train, seed=RANDOM_SEED)
selected_val_samples = sample_balanced(oi_val_view, need_add_val, seed=RANDOM_SEED + 1)

print(f"Selected train samples: {len(selected_train_samples)}")
print(f"Selected val samples  : {len(selected_val_samples)}")

Selected train samples: 9572
Selected val samples  : 2398


In [ ]:
def build_temp_dataset(name, samples):
    if fo.dataset_exists(name):
        fo.delete_dataset(name)
    ds = fo.Dataset(name)
    ds.add_samples(samples)
    return ds

sel_train_ds = build_temp_dataset("oi_selected_train_expand", selected_train_samples)
sel_val_ds = build_temp_dataset("oi_selected_val_expand", selected_val_samples)

 100% |███████████████| 9572/9572 [51.7s elapsed, 0s remaining, 247.9 samples/s]      


INFO:eta.core.utils: 100% |███████████████| 9572/9572 [51.7s elapsed, 0s remaining, 247.9 samples/s]      


 100% |███████████████| 2398/2398 [9.7s elapsed, 0s remaining, 236.8 samples/s]       


INFO:eta.core.utils: 100% |███████████████| 2398/2398 [9.7s elapsed, 0s remaining, 236.8 samples/s]       


In [ ]:
if OVERWRITE_EXISTING_EXPORT_TMP and os.path.exists(OI_EXPORT_DIR):
    shutil.rmtree(OI_EXPORT_DIR)

os.makedirs(OI_EXPORT_DIR, exist_ok=True)

sel_train_ds.export(
    export_dir=OI_EXPORT_DIR,
    dataset_type=fo.types.YOLOv5Dataset,
    label_field="ground_truth",
    split="train",
    classes=OI_CLASSES,
)

sel_val_ds.export(
    export_dir=OI_EXPORT_DIR,
    dataset_type=fo.types.YOLOv5Dataset,
    label_field="ground_truth",
    split="val",
    classes=OI_CLASSES,
)

print(f"\nExported YOLO data to: {OI_EXPORT_DIR}")

Directory '/content/openimages_expansion_tmp/oi_yolo_export' already exists; export will be merged with existing files


 100% |███████████████| 9572/9572 [1.4m elapsed, 0s remaining, 115.7 samples/s]      


INFO:eta.core.utils: 100% |███████████████| 9572/9572 [1.4m elapsed, 0s remaining, 115.7 samples/s]      


Directory '/content/openimages_expansion_tmp/oi_yolo_export' already exists; export will be merged with existing files


 100% |███████████████| 2398/2398 [18.3s elapsed, 0s remaining, 141.1 samples/s]      


INFO:eta.core.utils: 100% |███████████████| 2398/2398 [18.3s elapsed, 0s remaining, 141.1 samples/s]      



Exported YOLO data to: /content/openimages_expansion_tmp/oi_yolo_export


In [ ]:
OI_TO_TARGET_ID = {
    0: 0,
    1: 1,
    2: 2,
    3: 3,
    4: 4,
    5: 5,
}

def rewrite_label_file_inplace(label_path, remap):
    if not os.path.exists(label_path):
        return

    new_lines = []
    with open(label_path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    for line in lines:
        parts = line.strip().split()
        if len(parts) < 5:
            continue
        old_cls = int(float(parts[0]))
        if old_cls not in remap:
            continue
        parts[0] = str(remap[old_cls])
        new_lines.append(" ".join(parts))

    with open(label_path, "w", encoding="utf-8") as f:
        if new_lines:
            f.write("\n".join(new_lines) + "\n")
        else:
            f.write("")

export_train_lbls = glob.glob(os.path.join(OI_EXPORT_DIR, "train", "labels", "*.txt"))
export_val_lbls = glob.glob(os.path.join(OI_EXPORT_DIR, "val", "labels", "*.txt"))

for p in tqdm(export_train_lbls, desc="Remapping train labels"):
    rewrite_label_file_inplace(p, OI_TO_TARGET_ID)

for p in tqdm(export_val_lbls, desc="Remapping val labels"):
    rewrite_label_file_inplace(p, OI_TO_TARGET_ID)

Remapping train labels: 0it [00:00, ?it/s]

Remapping val labels: 0it [00:00, ?it/s]

In [ ]:
def safe_copy_append(src_img_dir, src_lbl_dir, dst_img_dir, dst_lbl_dir, prefix):
    img_exts = [".jpg", ".jpeg", ".png", ".bmp", ".webp"]
    copied = 0
    skipped_missing_label = 0
    skipped_empty_label = 0

    src_images = []
    for ext in img_exts:
        src_images.extend(glob.glob(os.path.join(src_img_dir, f"*{ext}")))

    for src_img in tqdm(src_images, desc=f"Appending {prefix} images"):
        stem = Path(src_img).stem
        suffix = Path(src_img).suffix.lower()
        src_lbl = os.path.join(src_lbl_dir, stem + ".txt")

        if not os.path.exists(src_lbl):
            skipped_missing_label += 1
            continue
        if os.path.getsize(src_lbl) == 0:
            skipped_empty_label += 1
            continue

        new_stem = f"{prefix}_{stem}"
        dst_img = os.path.join(dst_img_dir, new_stem + suffix)
        dst_lbl = os.path.join(dst_lbl_dir, new_stem + ".txt")

        shutil.copy2(src_img, dst_img)
        shutil.copy2(src_lbl, dst_lbl)
        copied += 1

    print(f"\n{prefix} summary:")
    print(f"  copied: {copied}")
    print(f"  skipped_missing_label: {skipped_missing_label}")
    print(f"  skipped_empty_label: {skipped_empty_label}")

    return copied

added_train = safe_copy_append(
    src_img_dir=os.path.join(OI_EXPORT_DIR, "images", "train"),
    src_lbl_dir=os.path.join(OI_EXPORT_DIR, "labels", "train"),
    dst_img_dir=train_img_dir,
    dst_lbl_dir=train_lbl_dir,
    prefix="oi_train"
)

added_val = safe_copy_append(
    src_img_dir=os.path.join(OI_EXPORT_DIR, "images", "val"),
    src_lbl_dir=os.path.join(OI_EXPORT_DIR, "labels", "val"),
    dst_img_dir=val_img_dir,
    dst_lbl_dir=val_lbl_dir,
    prefix="oi_val"
)

print(f"\nActually appended train images: {added_train}")
print(f"Actually appended val images  : {added_val}")

Appending oi_train images:   0%|          | 0/9572 [00:00<?, ?it/s]


oi_train summary:
  copied: 9572
  skipped_missing_label: 0
  skipped_empty_label: 0


Appending oi_val images:   0%|          | 0/2398 [00:00<?, ?it/s]


oi_val summary:
  copied: 2398
  skipped_missing_label: 0
  skipped_empty_label: 0

Actually appended train images: 9572
Actually appended val images  : 2398


In [ ]:
final_train_images, final_train_labels = count_images_labels(train_img_dir, train_lbl_dir)
final_val_images, final_val_labels = count_images_labels(val_img_dir, val_lbl_dir)

if os.path.exists(test_lbl_dir):
    final_test_images, final_test_labels = count_images_labels(test_img_dir, test_lbl_dir)
else:
    final_test_images = count_images_labels(test_img_dir, test_img_dir)[0]
    final_test_labels = 0

print("\nFinal dataset counts")
print(f"Train: {final_train_images} images, {final_train_labels} labels")
print(f"Val  : {final_val_images} images, {final_val_labels} labels")
print(f"Test : {final_test_images} images, {final_test_labels} labels")

assert final_test_images == current_test_images, "Test images changed, which should not happen."
assert final_test_labels == current_test_labels, "Test labels changed, which should not happen."

with open(YAML_PATH, "r", encoding="utf-8") as f:
    final_yaml = yaml.safe_load(f)

assert final_yaml == data_yaml, "YAML was changed unexpectedly."

print("\nSUCCESS: train/val expanded, test untouched, YAML unchanged.")
print("Your old training code should still work with the same dataset YAML path.")


Final dataset counts
Train: 14358 images, 14358 labels
Val  : 3597 images, 3597 labels
Test : 1015 images, 1015 labels

SUCCESS: train/val expanded, test untouched, YAML unchanged.
Your old training code should still work with the same dataset YAML path.


In [ ]:
print("Original train images:", current_train_images)
print("Final train images   :", final_train_images)
print("Original val images  :", current_val_images)
print("Final val images     :", final_val_images)
print("Test unchanged       :", final_test_images == current_test_images)

Original train images: 4786
Final train images   : 14358
Original val images  : 1199
Final val images     : 3597
Test unchanged       : True


In [ ]:
import os
import glob
from pathlib import Path
from collections import Counter, defaultdict
import yaml

IMG_EXTS = ("*.jpg", "*.jpeg", "*.png", "*.bmp", "*.webp")

EXPECTED_NAMES = {
    0: "person",
    1: "bicycle",
    2: "car",
    3: "motorcycle",
    4: "bus",
    5: "train",
}
EXPECTED_CLASS_IDS = set(EXPECTED_NAMES.keys())

def list_images(img_dir):
    files = []
    for ext in IMG_EXTS:
        files.extend(glob.glob(os.path.join(img_dir, ext)))
    return sorted(files)

def list_labels(lbl_dir):
    return sorted(glob.glob(os.path.join(lbl_dir, "*.txt")))

def stems_from_paths(paths):
    return [Path(p).stem for p in paths]

def find_duplicates(items):
    c = Counter(items)
    return {k: v for k, v in c.items() if v > 1}

def safe_getsize(path):
    try:
        return os.path.getsize(path)
    except Exception:
        return -1

def validate_yolo_label_file(path, expected_class_ids):
    """
    Returns a dict of issues found in a YOLO label file.
    YOLO detect rows should be: class x_center y_center width height
    """
    issues = {
        "missing": False,
        "empty": False,
        "bad_line_format": [],
        "bad_class_id": [],
        "bad_numeric": [],
        "out_of_range": [],
        "non_positive_wh": [],
        "line_count": 0,
        "class_counts": Counter(),
    }

    if not os.path.exists(path):
        issues["missing"] = True
        return issues

    if safe_getsize(path) == 0:
        issues["empty"] = True
        return issues

    with open(path, "r", encoding="utf-8") as f:
        lines = [ln.strip() for ln in f if ln.strip()]

    issues["line_count"] = len(lines)

    for i, line in enumerate(lines, start=1):
        parts = line.split()
        if len(parts) != 5:
            issues["bad_line_format"].append((i, line))
            continue

        cls_raw, x_raw, y_raw, w_raw, h_raw = parts

        # class id
        try:
            cls = int(float(cls_raw))
        except Exception:
            issues["bad_class_id"].append((i, cls_raw))
            continue

        if cls not in expected_class_ids:
            issues["bad_class_id"].append((i, cls))
            continue

        # box values
        try:
            x = float(x_raw)
            y = float(y_raw)
            w = float(w_raw)
            h = float(h_raw)
        except Exception:
            issues["bad_numeric"].append((i, line))
            continue

        # YOLO normalized values should be in [0,1]
        if not (0.0 <= x <= 1.0 and 0.0 <= y <= 1.0 and 0.0 <= w <= 1.0 and 0.0 <= h <= 1.0):
            issues["out_of_range"].append((i, line))

        if w <= 0 or h <= 0:
            issues["non_positive_wh"].append((i, line))

        issues["class_counts"][cls] += 1

    return issues

def inspect_split(name, img_dir, lbl_dir):
    images = list_images(img_dir)
    labels = list_labels(lbl_dir)

    image_stems = stems_from_paths(images)
    label_stems = stems_from_paths(labels)

    image_set = set(image_stems)
    label_set = set(label_stems)

    missing_labels = sorted(image_set - label_set)
    orphan_labels = sorted(label_set - image_set)

    dup_images = find_duplicates(image_stems)
    dup_labels = find_duplicates(label_stems)

    zero_imgs = [p for p in images if safe_getsize(p) == 0]
    zero_lbls = [p for p in labels if safe_getsize(p) == 0]

    bad_line_files = []
    bad_class_files = []
    bad_numeric_files = []
    out_of_range_files = []
    non_positive_wh_files = []

    total_boxes = 0
    class_counter = Counter()

    # validate only matched label files
    matched_stems = sorted(image_set & label_set)
    for stem in matched_stems:
        label_path = os.path.join(lbl_dir, stem + ".txt")
        issues = validate_yolo_label_file(label_path, EXPECTED_CLASS_IDS)

        total_boxes += sum(issues["class_counts"].values())
        class_counter.update(issues["class_counts"])

        if issues["bad_line_format"]:
            bad_line_files.append((stem, issues["bad_line_format"][:3]))
        if issues["bad_class_id"]:
            bad_class_files.append((stem, issues["bad_class_id"][:3]))
        if issues["bad_numeric"]:
            bad_numeric_files.append((stem, issues["bad_numeric"][:3]))
        if issues["out_of_range"]:
            out_of_range_files.append((stem, issues["out_of_range"][:3]))
        if issues["non_positive_wh"]:
            non_positive_wh_files.append((stem, issues["non_positive_wh"][:3]))

    ok = (
        len(missing_labels) == 0 and
        len(orphan_labels) == 0 and
        len(dup_images) == 0 and
        len(dup_labels) == 0 and
        len(zero_imgs) == 0 and
        len(zero_lbls) == 0 and
        len(bad_line_files) == 0 and
        len(bad_class_files) == 0 and
        len(bad_numeric_files) == 0 and
        len(out_of_range_files) == 0 and
        len(non_positive_wh_files) == 0
    )

    print(f"\n{'='*22} {name.upper()} {'='*22}")
    print(f"Images                         : {len(images)}")
    print(f"Labels                         : {len(labels)}")
    print(f"Matched pairs                  : {len(matched_stems)}")
    print(f"Missing labels                 : {len(missing_labels)}")
    print(f"Orphan labels                  : {len(orphan_labels)}")
    print(f"Duplicate image stems          : {len(dup_images)}")
    print(f"Duplicate label stems          : {len(dup_labels)}")
    print(f"Zero-byte images               : {len(zero_imgs)}")
    print(f"Zero-byte labels               : {len(zero_lbls)}")
    print(f"Files with bad row format      : {len(bad_line_files)}")
    print(f"Files with bad class IDs       : {len(bad_class_files)}")
    print(f"Files with bad numeric fields  : {len(bad_numeric_files)}")
    print(f"Files with out-of-range boxes  : {len(out_of_range_files)}")
    print(f"Files with non-positive w/h    : {len(non_positive_wh_files)}")
    print(f"Total labeled boxes            : {total_boxes}")
    print("Class counts (boxes):")
    for cid in sorted(EXPECTED_CLASS_IDS):
        print(f"  {cid} ({EXPECTED_NAMES[cid]}): {class_counter[cid]}")

    # sample issues
    def preview(title, items, n=5):
        if items:
            print(f"\n{title} (showing up to {n}):")
            for x in items[:n]:
                print(" ", x)

    preview("Missing labels", missing_labels)
    preview("Orphan labels", orphan_labels)
    preview("Duplicate image stems", list(dup_images.items()))
    preview("Duplicate label stems", list(dup_labels.items()))
    preview("Zero-byte images", zero_imgs)
    preview("Zero-byte labels", zero_lbls)
    preview("Bad row format files", bad_line_files)
    preview("Bad class ID files", bad_class_files)
    preview("Bad numeric files", bad_numeric_files)
    preview("Out-of-range box files", out_of_range_files)
    preview("Non-positive width/height files", non_positive_wh_files)

    print(f"\nSplit status: {'OK' if ok else 'PROBLEM FOUND'}")

    return {
        "name": name,
        "images": len(images),
        "labels": len(labels),
        "pairs": len(matched_stems),
        "missing_labels": missing_labels,
        "orphan_labels": orphan_labels,
        "dup_images": dup_images,
        "dup_labels": dup_labels,
        "zero_imgs": zero_imgs,
        "zero_lbls": zero_lbls,
        "bad_line_files": bad_line_files,
        "bad_class_files": bad_class_files,
        "bad_numeric_files": bad_numeric_files,
        "out_of_range_files": out_of_range_files,
        "non_positive_wh_files": non_positive_wh_files,
        "total_boxes": total_boxes,
        "class_counter": class_counter,
        "ok": ok,
    }

# ---- YAML sanity check ----
print("="*20, "YAML CHECK", "="*20)
yaml_ok = True
yaml_info = {}
try:
    with open(YAML_PATH, "r", encoding="utf-8") as f:
        y = yaml.safe_load(f)

    yaml_info = y
    print("YAML loaded successfully.")
    print("names:", y.get("names"))
    print("train:", y.get("train"))
    print("val  :", y.get("val"))
    print("test :", y.get("test"))

    if y.get("names") != EXPECTED_NAMES:
        yaml_ok = False
        print("YAML names mismatch.")
    else:
        print("YAML names mapping OK.")

except Exception as e:
    yaml_ok = False
    print("YAML load failed:", e)

# ---- Split checks ----
train_report = inspect_split("train", train_img_dir, train_lbl_dir)
val_report   = inspect_split("val", val_img_dir, val_lbl_dir)
test_report  = inspect_split("test", test_img_dir, test_lbl_dir)

# ---- Growth checks ----
print(f"\n{'='*20} GROWTH CHECK {'='*20}")
growth_ok = True
try:
    train_growth = train_report["images"] / current_train_images if current_train_images else None
    val_growth = val_report["images"] / current_val_images if current_val_images else None

    print(f"Current train before expansion : {current_train_images}")
    print(f"Current train now              : {train_report['images']}")
    print(f"Train growth factor            : {train_growth:.4f}" if train_growth is not None else "Train growth factor: N/A")

    print(f"Current val before expansion   : {current_val_images}")
    print(f"Current val now                : {val_report['images']}")
    print(f"Val growth factor              : {val_growth:.4f}" if val_growth is not None else "Val growth factor: N/A")

    # soft check: close to configured targets
    train_target = TRAIN_MULTIPLIER
    val_target = VAL_MULTIPLIER

    train_close = train_growth is not None and (train_growth >= train_target * 0.90)
    val_close = val_growth is not None and (val_growth >= val_target * 0.90)

    print(f"Train near target ({TRAIN_MULTIPLIER}x): {train_close}")
    print(f"Val near target   ({VAL_MULTIPLIER}x): {val_close}")

    if not (train_close and val_close):
        growth_ok = False
except Exception as e:
    growth_ok = False
    print("Growth check failed:", e)

# ---- Test unchanged ----
print(f"\n{'='*20} TEST UNCHANGED CHECK {'='*20}")
test_unchanged_ok = True
try:
    print(f"Original test images: {current_test_images}")
    print(f"Original test labels: {current_test_labels}")
    print(f"Current test images : {test_report['images']}")
    print(f"Current test labels : {test_report['labels']}")

    test_unchanged_ok = (
        test_report["images"] == current_test_images and
        test_report["labels"] == current_test_labels
    )
    print("Test unchanged:", test_unchanged_ok)
except Exception as e:
    test_unchanged_ok = False
    print("Test comparison failed:", e)

# ---- FiftyOne sanity ----
print(f"\n{'='*20} FIFTYONE CHECK {'='*20}")
fiftyone_ok = True
try:
    print(f"oi_train downloaded  : {len(oi_train)}")
    print(f"oi_val downloaded    : {len(oi_val)}")
    print(f"oi_train_view kept   : {len(oi_train_view)}")
    print(f"oi_val_view kept     : {len(oi_val_view)}")
    print(f"selected_train       : {len(selected_train_samples)}")
    print(f"selected_val         : {len(selected_val_samples)}")

    # expected selected counts should roughly equal what we wanted to add
    print(f"Needed add train     : {need_add_train}")
    print(f"Needed add val       : {need_add_val}")

    sel_train_ok = len(selected_train_samples) >= need_add_train
    sel_val_ok = len(selected_val_samples) >= need_add_val

    print(f"Selected train enough: {sel_train_ok}")
    print(f"Selected val enough  : {sel_val_ok}")

    fiftyone_ok = sel_train_ok and sel_val_ok
except Exception as e:
    fiftyone_ok = False
    print("FiftyOne check failed:", e)

# ---- Global verdict ----
all_ok = (
    yaml_ok and
    train_report["ok"] and
    val_report["ok"] and
    test_report["ok"] and
    growth_ok and
    test_unchanged_ok and
    fiftyone_ok
)

print(f"\n{'='*20} FINAL VERDICT {'='*20}")
print("YAML OK             :", yaml_ok)
print("TRAIN OK            :", train_report["ok"])
print("VAL OK              :", val_report["ok"])
print("TEST OK             :", test_report["ok"])
print("GROWTH OK           :", growth_ok)
print("TEST UNCHANGED OK   :", test_unchanged_ok)
print("FIFTYONE OK         :", fiftyone_ok)
print("\nOVERALL DATASET STATUS:", "SAFE / CONSISTENT" if all_ok else "ISSUES FOUND")

if all_ok:
    print("\nYou can proceed to training with higher confidence.")
else:
    print("\nDo not start training yet. Inspect the sections above and fix the reported issues first.")

==================== YAML CHECK ====================
YAML loaded successfully.
names: {0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 4: 'bus', 5: 'train'}
train: train/images
val  : val/images
test : test/images
YAML names mapping OK.

====================== TRAIN ======================
Images                         : 14358
Labels                         : 14358
Matched pairs                  : 14358
Missing labels                 : 0
Orphan labels                  : 0
Duplicate image stems          : 0
Duplicate label stems          : 0
Zero-byte images               : 0
Zero-byte labels               : 0
Files with bad row format      : 0
Files with bad class IDs       : 0
Files with bad numeric fields  : 0
Files with out-of-range boxes  : 0
Files with non-positive w/h    : 0
Total labeled boxes            : 43144
Class counts (boxes):
  0 (person): 14322
  1 (bicycle): 3879
  2 (car): 20092
  3 (motorcycle): 1754
  4 (bus): 1541
  5 (train): 1556

Split status: OK

========

In [ ]:
import os

def show_tree(root, max_depth=3, max_items=20):
    print(f"\nTREE: {root}")
    if not os.path.exists(root):
        print("Path does not exist")
        return

    root = os.path.abspath(root)
    for current_root, dirs, files in os.walk(root):
        rel = os.path.relpath(current_root, root)
        depth = 0 if rel == "." else rel.count(os.sep) + 1
        if depth > max_depth:
            dirs[:] = []
            continue

        indent = "    " * depth
        print(f"{indent}[{os.path.basename(current_root)}]")

        shown = 0
        for f in sorted(files):
            print(f"{indent}    {f}")
            shown += 1
            if shown >= max_items:
                remaining = len(files) - shown
                if remaining > 0:
                    print(f"{indent}    ... ({remaining} more files)")
                break

show_tree(OI_EXPORT_DIR, max_depth=4, max_items=10)


TREE: /content/openimages_expansion_tmp/oi_yolo_export
[oi_yolo_export]
    dataset.yaml
    [labels]
        [train]
            0001df0030045d30.txt
            0002262376683c96.txt
            0003d828877c9807.txt
            0006176101d9d341.txt
            000780f791b71252.txt
            0008104bd27dd02a.txt
            00088e25af342aa4.txt
            0009b74be294a699.txt
            000b0f235dcf2caa.txt
            000bcee5bed5446b.txt
            ... (9562 more files)
        [val]
            001a794d1865ee47.txt
            001a809ad40a2f84.txt
            002d1dd67c722d98.txt
            002f8241bd829022.txt
            005ea49186b013d8.txt
            0082b95745cb8ed5.txt
            00a1f2a6e7f78ac5.txt
            00acf53b127218c2.txt
            00e9084d1bc8e0ea.txt
            0104cba3ff87376c.txt
            ... (2388 more files)
    [images]
        [train]
            0001df0030045d30.jpg
            0002262376683c96.jpg
            0003d828877c9807.jpg
           

In [ ]:
import os
import shutil
from google.colab import files

SOURCE_FOLDER = "/content/drive/MyDrive/Lab thầy Vinh/Train_VOC_Test_Dawn/Train_VOC_Test_Dawn"
LOCAL_PARENT = "/content"
LOCAL_FOLDER = os.path.join(LOCAL_PARENT, "Train_VOC_Test_Dawn")
ZIP_BASE = os.path.join(LOCAL_PARENT, "Train_VOC_Test_Dawn")
ZIP_PATH = ZIP_BASE + ".zip"

# Clean old local copy / zip
if os.path.exists(LOCAL_FOLDER):
    shutil.rmtree(LOCAL_FOLDER)
if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)

print("Copying dataset from Drive to local Colab storage...")
shutil.copytree(SOURCE_FOLDER, LOCAL_FOLDER)

print("Creating zip locally...")
shutil.make_archive(
    base_name=ZIP_BASE,
    format="zip",
    root_dir=LOCAL_PARENT,
    base_dir="Train_VOC_Test_Dawn",
)

size_mb = os.path.getsize(ZIP_PATH) / (1024 ** 2)
print(f"Created: {ZIP_PATH}")
print(f"Size: {size_mb:.2f} MB")

print("Starting download...")
files.download(ZIP_PATH)

Copying dataset from Drive to local Colab storage...
Creating zip locally...
Created: /content/Train_VOC_Test_Dawn.zip
Size: 5241.28 MB
Starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>